In [2]:
import requests 
import csv
from bs4 import BeautifulSoup
import pandas as pd
import matplotlib.pyplot as plt 
from wordcloud import WordCloud
from urllib.parse import urljoin

import time
import datetime 
from datetime import datetime, date, timedelta, timezone 
from collections import Counter


In [10]:
#the site's root address. Capital letters show "constant, don't change"
BASE_URL = "https://books.toscrape.com/"
#the User-Agent tells the server what is visiting, without it some sites could block your request if its not sent
HEADERS = {"User-Agent": "BooksScraper/1.0"}

#seconds between request
DELAY = 1

#this is a safety switch. At 20 the script only scrapes 20 books, for a quick test 
#LIMIT = 20
#scraping all the books
LIMIT = None
#ratings are stored as words, so we are mapping them to numbers 
RATINGS = {"One":1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

In [11]:
#creates a session which keeps the connection open and reuses it without staring a 
#new connection with every page
session = requests.Session()
#attached headers to the session, so every request sends them automatically
session.headers.update(HEADERS)


In [12]:
#this is the safe downloader every page request in the script goes through it
def get_soup(url, retries=3):
    """Download and parse a page, retrying with back-off on failure."""
    #is has three retries to downlaod and parse a page incase of failure 
    for attempt in range(1, retries + 1):
        try:
            r = session.get(url, timeout=15)
            #this turns errors like 404 into exceptions, then the except catches them also
            r.raise_for_status()
            r.encoding = "utf-8"
            return BeautifulSoup(r.text, "html.parser")
        except requests.RequestException as e:
            # 2s, 4s, 8s
            #if there's a failure it waits 2 seconds to try again, then 4 then 8 
            wait = 2 ** attempt          
            print(f"  Error on {url}: {e}. Retry {attempt}/{retries} in {wait}s")
            time.sleep(wait)
    # give up on this URL, don't crash
    #if all three attempts fail it skips the page and carries on
    return None                          


In [13]:
#collecting the address of every book 
def get_book_urls():
    """Walk all listing pages and return every book's detail-page URL."""
    urls = []
    page_url = urljoin(BASE_URL, "catalogue/page-1.html")
    while page_url:
        soup = get_soup(page_url)
        if soup is None:
            break
        #this finds the link to each of the 20 books on the page 
        for a in soup.select("article.product_pod h3 a"):
            urls.append(urljoin(page_url, a["href"]))
        next_link = soup.select_one("li.next a")
        #urljoin turns each relative link into a full url
        #after that it looks for the next button, if one exists, it visits that next page
        #if not page_url becomes None and the while loop ends; pagination
        page_url = urljoin(page_url, next_link["href"]) if next_link else None
        print(f"Collected {len(urls)} URLs")
        time.sleep(DELAY)
    return urls



In [14]:
def parse_book(soup, url):
    """Extract all fields from one book detail page."""
    #this loops over every row and builds a dictionary, since each row is a label and a value
    table = {row.th.text: row.td.text for row in soup.select("table.table-striped tr")}

    #p is for the description 
    desc_header = soup.select_one("#product_description")
    #handling books without descriptions 
    description = desc_header.find_next_sibling("p").text if desc_header else ""

    rating_tag = soup.select_one("p.star-rating")
    rating = RATINGS.get(rating_tag["class"][1]) if rating_tag else None

    #a dictionary with one entry per field 
    return {
        "title": soup.h1.text,
        "category": soup.select("ul.breadcrumb li a")[-1].text,
        #returns None if the key doesn't exists
        "upc": table.get("UPC"),
        #.get() is used when data might be missing
        "product_type": table.get("Product Type"),
        "price_excl_tax": table.get("Price (excl. tax)"),
        "price_incl_tax": table.get("Price (incl. tax)"),
        "tax": table.get("Tax"),
        "availability": table.get("Availability"),
        "num_reviews": table.get("Number of reviews"),
        "rating": rating,
        "description": description,
        "image_url": urljoin(url, soup.select_one("div.item.active img")["src"]),
        "url": url,
        #this stores the current UTC(Coordinated Universal Time) time; it gives the timestamp
        "scraped_at": datetime.now(timezone.utc).isoformat(),
    }


In [15]:
def main():
    #getting all the urls, and cutting the list down since the limit was set to 20
    urls = get_book_urls()
    if LIMIT:
        urls = urls[:LIMIT]

    books, failed = [], []
    #enumerate helps us list the urls starting from 1
    for i, url in enumerate(urls, start=1):
        soup = get_soup(url)
        #if the download failed, the URL goes into failed and continue to the next
        if soup is None:
            failed.append(url)
            continue
        #this section catches layout changes. If one page is built differently and parse_book crashes, only that book is skipped
        try:
            books.append(parse_book(soup, url))
        except Exception as e:           # layout surprise: log it, keep going
            print(f"  Parse error on {url}: {e}")
            failed.append(url)
        #this is for getting a progress message after every 50 books that have been scrapped
        if i % 50 == 0:
            print(f"Scraped {i}/{len(urls)} books")

        #a quick pause, before moving to the next request
        time.sleep(DELAY)
    #writing everything in CSV, reporting how many succeeded and how many failed
    with open("books_raw.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=books[0].keys())
        writer.writeheader()
        writer.writerows(books)

    print(f"Done: {len(books)} books saved, {len(failed)} failed")
    if failed:
        print("Failed URLs:", failed)

In [16]:
if __name__ == "__main__":
    main()

Collected 20 URLs
Collected 40 URLs
Collected 60 URLs
Collected 80 URLs
Collected 100 URLs
Collected 120 URLs
Collected 140 URLs
Collected 160 URLs
Collected 180 URLs
Collected 200 URLs
Collected 220 URLs
Collected 240 URLs
Collected 260 URLs
Collected 280 URLs
Collected 300 URLs
Collected 320 URLs
Collected 340 URLs
Collected 360 URLs
Collected 380 URLs
Collected 400 URLs
Collected 420 URLs
Collected 440 URLs
Collected 460 URLs
Collected 480 URLs
Collected 500 URLs
Collected 520 URLs
Collected 540 URLs
Collected 560 URLs
Collected 580 URLs
Collected 600 URLs
Collected 620 URLs
Collected 640 URLs
Collected 660 URLs
Collected 680 URLs
Collected 700 URLs
Collected 720 URLs
Collected 740 URLs
Collected 760 URLs
Collected 780 URLs
Collected 800 URLs
Collected 820 URLs
Collected 840 URLs
Collected 860 URLs
Collected 880 URLs
Collected 900 URLs
Collected 920 URLs
Collected 940 URLs
Collected 960 URLs
Collected 980 URLs
Collected 1000 URLs
Scraped 50/1000 books
Scraped 100/1000 books
Scraped 

In [9]:
import pandas as pd
df = pd.read_csv("books_raw.csv")
# expect (1000, 14)
print(df.shape)   
# expect 1000, so no duplicates                   
print(df["upc"].nunique())           
print(df.isna().sum())     
# look for missing fields          
print(df["category"].value_counts().head())

(20, 14)
20
title             0
category          0
upc               0
product_type      0
price_excl_tax    0
price_incl_tax    0
tax               0
availability      0
num_reviews       0
rating            0
description       0
image_url         0
url               0
scraped_at        0
dtype: int64
category
Poetry                4
Default               3
Young Adult           2
Music                 2
Historical Fiction    1
Name: count, dtype: int64
